In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os
import random
import argparse
import csv
import math
import sys
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

In [2]:
!mkdir -p "/content/hw4-data-release"

In [6]:
!unzip "/content/drive/MyDrive/Visual Recognition/hw4/hw4_realse_dataset.zip" -d "/content/hw4-data-release"

Streaming output truncated to the last 5000 lines.
  inflating: /content/hw4-data-release/hw4_realse_dataset/train/clean/snow_clean-1133.png  
  inflating: /content/hw4-data-release/__MACOSX/hw4_realse_dataset/train/clean/._snow_clean-1133.png  
  inflating: /content/hw4-data-release/hw4_realse_dataset/train/clean/snow_clean-871.png  
  inflating: /content/hw4-data-release/__MACOSX/hw4_realse_dataset/train/clean/._snow_clean-871.png  
  inflating: /content/hw4-data-release/hw4_realse_dataset/train/clean/rain_clean-931.png  
  inflating: /content/hw4-data-release/__MACOSX/hw4_realse_dataset/train/clean/._rain_clean-931.png  
  inflating: /content/hw4-data-release/hw4_realse_dataset/train/clean/snow_clean-124.png  
  inflating: /content/hw4-data-release/__MACOSX/hw4_realse_dataset/train/clean/._snow_clean-124.png  
  inflating: /content/hw4-data-release/hw4_realse_dataset/train/clean/rain_clean-702.png  
  inflating: /content/hw4-data-release/__MACOSX/hw4_realse_dataset/train/clean/._rai

In [5]:
data_path = "/content/hw4-data-release/hw4_realse_dataset"

In [20]:
RAIN_LABEL = 0
SNOW_LABEL = 1
LABEL_TO_NAME = {RAIN_LABEL: "rain", SNOW_LABEL: "snow"}

def convert_to_tensor(img):
    arr = np.asarray(img.convert("RGB"), dtype=np.uint8)
    tensor = torch.from_numpy(arr).permute(2, 0, 1).contiguous().float().div_(255.0)
    return tensor


def random_crop(degraded, clean, patch_size):
    _, h, w = degraded.shape
    if h < patch_size or w < patch_size:
        pad_h = max(0, patch_size - h)
        pad_w = max(0, patch_size - w)

        degraded = torch.nn.functional.pad(degraded, (0, pad_w, 0, pad_h), mode="reflect")
        clean = torch.nn.functional.pad(clean, (0, pad_w, 0, pad_h), mode="reflect")

        _, h, w = degraded.shape

    top = random.randint(0, h - patch_size)
    left = random.randint(0, w - patch_size)

    degraded = degraded[:, top : top + patch_size, left : left + patch_size]
    clean = clean[:, top : top + patch_size, left : left + patch_size]

    return degraded, clean


def random_flip(degraded, clean):
    if random.random() < 0.5:
        degraded = torch.flip(degraded, dims=[2])
        clean = torch.flip(clean, dims=[2])

    if random.random() < 0.5:
        degraded = torch.flip(degraded, dims=[1])
        clean = torch.flip(clean, dims=[1])

    return degraded, clean



@dataclass(frozen=True)
class PairedSample:
    degraded_path: str
    clean_path: str
    label: int  # 0 = rain, 1 = snow


def perform_pair_indexing(data_root):

    degraded_dir = Path(data_root) / "train" / "degraded"
    clean_dir = Path(data_root) / "train" / "clean"

    if not degraded_dir.is_dir() or not clean_dir.is_dir():
        raise FileNotFoundError(f"Wrong dataset format!")

    samples: list[PairedSample] = []

    for fname in sorted(os.listdir(degraded_dir)):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        stem, ext = os.path.splitext(fname)
        if stem.startswith("rain-"):
            label = RAIN_LABEL
            idx = stem[len("rain-"):]
            clean_name = f"rain_clean-{idx}{ext}"
        elif stem.startswith("snow-"):
            label = SNOW_LABEL
            idx = stem[len("snow-"):]
            clean_name = f"snow_clean-{idx}{ext}"
        else:
            continue

        deg_path = degraded_dir / fname
        clean_path = clean_dir / clean_name

        if not clean_path.is_file():
            raise FileNotFoundError(f"Missing '{deg_path}' clean instance!")

        samples.append(
            PairedSample(
                degraded_path=str(deg_path),
                clean_path=str(clean_path),
                label=label,
            )
        )

    return samples


def perform_test_file_indexing(data_root):
    test_dir = Path(data_root) / "test" / "degraded"

    if not test_dir.is_dir():
        raise FileNotFoundError(f"Directory not found: {test_dir}")

    files = sorted(
        p for p in test_dir.iterdir()
        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )

    if not files:
        raise RuntimeError(f"No test images found under {test_dir}")

    return [str(p) for p in files] # to return str not Path

class MyTrainDataset(Dataset):
    def __init__(self, samples, patch_size = 128, augment = True):
        super().__init__()
        self.samples = samples
        self.patch_size = patch_size
        self.augment = augment

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        degraded = convert_to_tensor(Image.open(sample.degraded_path))
        clean = convert_to_tensor(Image.open(sample.clean_path))

        degraded, clean = random_crop(degraded, clean, self.patch_size)

        if self.augment:
            degraded, clean = random_flip(degraded, clean)

        return {
            "degraded": degraded,
            "clean": clean,
            "label": torch.tensor(sample.label, dtype=torch.long),
        }


class MyValidationDataset(Dataset):
    def __init__(self, samples):
        super().__init__()
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx) -> dict[str, torch.Tensor]:
        sample = self.samples[idx]
        degraded = convert_to_tensor(Image.open(sample.degraded_path))
        clean = convert_to_tensor(Image.open(sample.clean_path))

        return {
            "degraded": degraded,
            "clean": clean,
            "label": torch.tensor(sample.label, dtype=torch.long),
        }


class MyTestDataset(Dataset):
    def __init__(self, file_paths):
        super().__init__()
        self.file_paths = file_paths

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int) -> dict:
        path = self.file_paths[idx]
        degraded = convert_to_tensor(Image.open(path))

        return {
            "degraded": degraded,
            "filename": os.path.basename(path),
        }

def perform_dataset_split(samples, val_ratio, seed):
    rng = random.Random(seed)
    by_label: dict[int, list[PairedSample]] = {RAIN_LABEL: [], SNOW_LABEL: []}
    for s in samples:
        by_label[s.label].append(s)

    train: list[PairedSample] = []
    val: list[PairedSample] = []

    for label, group in by_label.items():
        group = list(group)
        rng.shuffle(group)

        n_val = int(round(len(group) * val_ratio))
        val.extend(group[:n_val])
        # train.extend(group[n_val:])
        train.extend(group)

    rng.shuffle(train)
    return train, val

In [21]:
def _test_collate(batch: list[dict]) -> dict:
    return {
        "degraded": torch.stack([item["degraded"] for item in batch], dim=0),
        "filename": [item["filename"] for item in batch],
    }

def build_dataloaders(
    data_root,
    patch_size = 128,
    batch_size = 8,
    val_ratio = 0.1,
    num_workers = 2,
    seed = 42,
    include_test = True,
):

    all_pairs = perform_pair_indexing(data_root)
    train_pairs, val_pairs = perform_dataset_split(all_pairs, val_ratio, seed)

    train_ds = MyTrainDataset(train_pairs, patch_size=patch_size, augment=True)
    val_ds = MyValidationDataset(val_pairs)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=num_workers > 0,
        # prefetch_factor=4,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        # prefetch_factor=4,
    )

    test_loader = None
    test_size = 0

    if include_test:
        test_files = perform_test_file_indexing(data_root)
        test_ds = MyTestDataset(test_files)

        test_loader = DataLoader(
            test_ds,
            batch_size=1,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True,
            collate_fn=_test_collate,
            persistent_workers=num_workers > 0,
            # prefetch_factor=4,
        )
        test_size = len(test_ds)

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "test_size": test_size,
    }

In [9]:
!git clone https://github.com/va1shn9v/PromptIR.git /content/PromptIR

Cloning into '/content/PromptIR'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 126 (delta 45), reused 37 (delta 37), pack-reused 70 (from 1)
Receiving objects: 100% (126/126), 1.39 MiB | 36.41 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [10]:
!pip -q install einops

In [11]:
promptIR_path = "/content/PromptIR"

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def import_promptir(repo_path):
    repo = Path(repo_path).resolve()

    if not repo.is_dir():
        raise FileNotFoundError(
            f"PromptIR repo not found at: {repo}"
        )

    model_file = repo / "net" / "model.py"

    if not model_file.is_file():
        raise FileNotFoundError(
            f"{model_file} not found."
        )

    sys.path.insert(0, str(repo))
    from net.model import PromptIR
    return PromptIR


def compute_psnr(pred, target):
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2).item()

    if mse <= 1e-12:
        return 100.0
    return 10.0 * math.log10(1.0 / mse)


class MyAverager:
    def __init__(self):
        self.sum = 0.0
        self.count = 0

    def update(self, value, n = 1):
        self.sum += value * n
        self.count += n

    @property
    def avg(self) -> float:
        return self.sum / self.count if self.count > 0 else 0.0


def train_one_epoch(model, loader, optimizer, scaler, loss_fn, device, epoch, log_every, use_amp,):

    model.train()
    meter = MyAverager()
    t0 = time.time()

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch:03d}",
        leave=False,
        dynamic_ncols=True,
    )

    for step, batch in enumerate(pbar):
        degraded = batch["degraded"].to(device, non_blocking=True)
        clean = batch["clean"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            restored = model(degraded)
            loss = loss_fn(restored, clean)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        meter.update(loss.item(), n=degraded.size(0))
        pbar.set_postfix({
            "Carbonier":  f"{meter.avg:.4f}",
            "lr":  f"{optimizer.param_groups[0]['lr']:.2e}",
        })

        if (step + 1) % log_every == 0 or (step + 1) == len(loader):
            elapsed = time.time() - t0
            print(
                f"  epoch {epoch:03d} | step {step + 1:4d}/{len(loader)} "
                f"| Carbonier {meter.avg:.4f} | {elapsed:6.1f}s",
                flush=True,
            )
    return meter.avg


@torch.no_grad()
def validate(
    model,
    loader,
    device,
    use_amp,
):
    model.eval()
    psnr_by_label: dict[int, list[float]] = {0: [], 1: []}

    pbar = tqdm(loader, desc="Validating", leave=False, dynamic_ncols=True)
    for batch in pbar:
        degraded = batch["degraded"].to(device, non_blocking=True)
        clean = batch["clean"].to(device, non_blocking=True)
        label = int(batch["label"].item())

        with torch.cuda.amp.autocast(enabled=use_amp):
            restored = model(degraded)

        psnr = compute_psnr(restored.float(), clean.float())
        psnr_by_label[label].append(psnr)

    rain_psnr = float(np.mean(psnr_by_label[0])) if psnr_by_label[0] else 0.0
    snow_psnr = float(np.mean(psnr_by_label[1])) if psnr_by_label[1] else 0.0
    all_vals = psnr_by_label[0] + psnr_by_label[1]
    overall_psnr = float(np.mean(all_vals)) if all_vals else 0.0
    return {
        "psnr_rain": rain_psnr,
        "psnr_snow": snow_psnr,
        "psnr_overall": overall_psnr,
        "n_rain": len(psnr_by_label[0]),
        "n_snow": len(psnr_by_label[1]),
    }




def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_psnr,
):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if scaler is not None else None,
            "best_psnr": best_psnr,
        },
        path,
    )


def resumer(ckpt_path, model, optimizer, scheduler, scaler, device):
    if not ckpt_path.is_file():
        return 0, -1.0

    print(f"Resuming from {ckpt_path}")
    state = torch.load(ckpt_path, map_location=device)

    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])

    if scaler is not None and state.get("scaler") is not None:
        scaler.load_state_dict(state["scaler"])

    start_epoch = int(state["epoch"]) + 1
    best_psnr = float(state.get("best_psnr", -1.0))

    print(f"  resumed at epoch {start_epoch}, best PSNR so far {best_psnr:.3f}")
    return start_epoch, best_psnr


class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3) :
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff * diff + self.eps ** 2))

def main():
    data_root = data_path # = "/content/hw4-data-release/hw4_realse_dataset"
    promptir_repo = promptIR_path # = "/content/PromptIR"
    out_dir = "/content/drive/MyDrive/Visual Recognition/hw4"

    patch_size = 256
    batch_size = 1
    val_ratio = 0.1
    num_workers = 8

    epochs = 50
    lr = 2e-4
    weight_decay = 1e-4
    min_lr = 1e-6
    grad_clip = 1.0

    val_every = 1
    log_every = 200

    no_amp = False
    resume = False
    seed = 42

    set_seed(seed)

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    best_ckpt = out_dir / "best.pt"
    last_ckpt = out_dir / "last.pt"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("WARNING: no CUDA device found.")

    torch.backends.cudnn.benchmark = True
    use_amp = (not no_amp) and device.type == "cuda"

    PromptIR = import_promptir(promptir_repo)
    model = PromptIR(decoder=True).to(device)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model: PromptIR(decoder=True)  ({n_params:.2f}M params)")

    print(f"Building dataloaders from: {data_root}")
    print(f"Patch size = {patch_size}")
    loaders = build_dataloaders(
        data_root=data_root,
        patch_size=patch_size,
        batch_size=batch_size,
        val_ratio=val_ratio,
        num_workers=num_workers,
        seed=seed,
        include_test=False,
    )
    train_loader = loaders["train_loader"]
    val_loader = loaders["val_loader"]

    print(
        f"  train batches: {len(train_loader)} "
        f"({loaders['train_size']} samples)  "
        f"| val batches: {len(val_loader)} ({loaders['val_size']} samples)"
    )

    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
    )

    scheduler = CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=min_lr
    )

    # loss_fn = nn.L1Loss()
    loss_fn = CharbonnierLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    start_epoch = 0
    best_psnr = -1.0
    if resume:
        start_epoch, best_psnr = resumer(
            last_ckpt, model, optimizer, scheduler, scaler, device
        )

    print(f"Starting training: epochs {start_epoch} to {epochs - 1}")

    for epoch in range(start_epoch, epochs):


        t0 = time.time()
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"\n=== Epoch {epoch:03d}  (lr={lr_now:.2e}) ===")


        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            loss_fn=loss_fn,
            device=device,
            epoch=epoch,
            log_every=log_every,
            use_amp=use_amp,
        )

        scheduler.step()

        val_metrics = {"psnr_rain": 0.0, "psnr_snow": 0.0, "psnr_overall": 0.0}

        if (epoch + 1) % val_every == 0 or epoch == epochs - 1:
            val_metrics = validate(model, val_loader, device, use_amp)
            print(
                f"  val PSNR | rain {val_metrics['psnr_rain']:.3f} "
                f"({val_metrics['n_rain']}) "
                f"| snow {val_metrics['psnr_snow']:.3f} "
                f"({val_metrics['n_snow']}) "
                f"| overall {val_metrics['psnr_overall']:.3f}"
            )

            if val_metrics["psnr_overall"] > best_psnr:
                best_psnr = val_metrics["psnr_overall"]
                save_checkpoint(
                    best_ckpt, model, optimizer, scheduler,
                    scaler, epoch, best_psnr
                )
                print(f"  >> NEW BEST val PSNR = {best_psnr:.3f}  "
                    f"(saved to {best_ckpt})")


        save_checkpoint(
            last_ckpt, model, optimizer, scheduler,
            scaler, epoch, best_psnr
        )

        epoch_sec = time.time() - t0

    print(f"\nDone. Best val PSNR: {best_psnr:.3f}  ({best_ckpt})")


if __name__ == "__main__":
    main()

In [ ]:
CKPT_PATH   = "/content/drive/MyDrive/Visual Recognition/hw4/best.pt"
OUT_NPZ     = "/content/drive/MyDrive/Visual Recognition/hw4/pred.npz"
DATA_ROOT   = data_path
PROMPT_REPO = promptIR_path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PromptIR = import_promptir(PROMPT_REPO)
model = PromptIR(decoder=True).to(device)
state = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(state["model"])
model.eval()
print(f"Loaded checkpoint from epoch {state['epoch']}, "
      f"best val PSNR = {state['best_psnr']:.3f}")

loaders = build_dataloaders(
    data_root=DATA_ROOT,
    patch_size=256,
    batch_size=1,
    val_ratio=0.1,
    num_workers=8,
    seed=42,
    include_test=True,
)
test_loader = loaders["test_loader"]
print(f"Test images: {loaders['test_size']}")


def perform_x4_tta(model, x):
    augments = [
        lambda t: t,
        lambda t: torch.flip(t, dims=[3]),
        lambda t: torch.flip(t, dims=[2]),
        lambda t: torch.flip(t, dims=[2, 3]),
    ]

    inverses = augments

    preds = []
    for aug, inv in zip(augments, inverses):
        x_aug = aug(x)
        y_aug = model(x_aug)
        preds.append(inv(y_aug))
    return torch.stack(preds, dim=0).mean(dim=0)


images_dict: dict[str, np.ndarray] = {}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference (TTA)", dynamic_ncols=True):
        degraded = batch["degraded"].to(device, non_blocking=True)
        filename = batch["filename"][0]

        restored = perform_x4_tta(model, degraded)

        restored = restored.clamp(0.0, 1.0).squeeze(0).cpu()
        arr = (restored * 255.0).round().to(torch.uint8).numpy()
        images_dict[filename] = arr

Path(OUT_NPZ).parent.mkdir(parents=True, exist_ok=True)
np.savez(OUT_NPZ, **images_dict)
print(f"Saved {len(images_dict)} restored images to {OUT_NPZ}")